# Qwen3-4B manual Kaggle / Mac pilot
Private notebook. Qwen inference runs here on one GPU; task commands run only in disposable Docker on the Mac. This is a separate platform condition after Colab GPU quota interruption. Same pinned checkpoint/prompts/generation, upstream ten-error case, twelve calls/90 minutes. Follow docs/KAGGLE_MANUAL_RUN.md. Do not Run All or regenerate a completed call. No Google Drive mount or Colab imports are used.


In [ ]:
import torch, platform
from packaging.version import Version
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings before proceeding'
assert Version('2.4') <= Version(torch.__version__.split('+')[0]) < Version('3'), 'Use a current Kaggle GPU image; do not replace CUDA Torch blindly'
torch.cuda.set_device(0)
print('Python:', platform.python_version())
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
print('Visible GPUs:', [(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])
print('Only GPU 0 is used; multiple GPUs do not pool memory in this runner.')


## Attach the package and verify file transfer first
Enable Internet for dependency/model downloads. Attach work/qwen3_hybrid_kaggle_20261101.zip as a PRIVATE Kaggle input; do not publish experiment files. Kaggle may expand the ZIP into a folder, which the verifier supports. Run the setup cell once, then download its small transfer-check file before loading the model. GPU/quota availability and the file-download UI are live checks, not assumed capabilities.


In [ ]:
from pathlib import Path
import hashlib, json, sys, subprocess, zipfile, stat, shutil
from IPython.display import display, FileLink

PACKAGE_NAME = 'qwen3_hybrid_kaggle_20261101.zip'
CONFIG_NAME = 'configs/precommit_hook_qwen3_hybrid_small_kaggle.json'
ROOT = Path('/kaggle/working/capstone/ai-safety-capstone')

def safe_relative(name):
    path = Path(name)
    if path.is_absolute() or '..' in path.parts or chr(92) in name or not path.parts:
        raise ValueError('Unsafe package path')
    return path

def install_package(source, destination):
    source, destination = Path(source), Path(destination)
    if destination.is_symlink():
        raise ValueError('Package destination must not be a link')
    if source.is_file():
        if source.stat().st_size > 10_000_000:
            raise ValueError('Unexpected package size; do not upload weights')
        with zipfile.ZipFile(source) as archive:
            for item in archive.infolist():
                safe_relative(item.filename)
                if stat.S_ISLNK(item.external_attr >> 16):
                    raise ValueError('Package links unsupported')
            manifest_bytes = archive.read('ai-safety-capstone/PACKAGE_MANIFEST.json')
            manifest = json.loads(manifest_bytes)
            payloads = {name: archive.read('ai-safety-capstone/' + str(safe_relative(name)))
                        for name in manifest['files_sha256']}
        archive_hash = hashlib.sha256(source.read_bytes()).hexdigest()
    else:
        manifest_bytes = (source/'PACKAGE_MANIFEST.json').read_bytes()
        manifest = json.loads(manifest_bytes)
        payloads = {}
        for name in manifest['files_sha256']:
            path = source/safe_relative(name)
            if path.is_symlink() or not path.resolve().is_relative_to(source.resolve()):
                raise ValueError('Package source link/escape unsupported')
            payloads[name] = path.read_bytes()
        archive_hash = None
    for name, content in payloads.items():
        if name == 'PACKAGE_MANIFEST.json' or hashlib.sha256(content).hexdigest() != manifest['files_sha256'][name]:
            raise ValueError('Package source checksum mismatch: ' + name)
    if destination.exists():
        if json.loads((destination/'PACKAGE_MANIFEST.json').read_text()) != manifest:
            raise ValueError('Existing package differs; do not overlay')
        for name, content in payloads.items():
            if (destination/safe_relative(name)).read_bytes() != content:
                raise ValueError('Existing source differs: ' + name)
    else:
        destination.mkdir(parents=True)
        for name, content in payloads.items():
            target = destination/safe_relative(name)
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open('xb') as stream:
                stream.write(content)
        with (destination/'PACKAGE_MANIFEST.json').open('xb') as stream:
            stream.write(manifest_bytes)
    return manifest, {'archive_sha256': archive_hash,
        'manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(),
        'input_mode': 'zip' if source.is_file() else 'kaggle_expanded_input'}

zip_inputs = sorted(Path('/kaggle/input').rglob(PACKAGE_NAME))
folder_inputs = sorted(p.parent for p in Path('/kaggle/input').rglob('PACKAGE_MANIFEST.json')
                       if (p.parent/CONFIG_NAME).is_file())
choices = zip_inputs if zip_inputs else folder_inputs
assert len(choices) == 1, 'Attach exactly one matching private package input; found: ' + str(choices)
manifest, package_identity = install_package(choices[0], ROOT)
print('Verified package files:', len(manifest['files_sha256']))
print('Package identity:', package_identity)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-thinking.txt')], check=True)
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
probe = ROOT/'work/kaggle_transfer_check.json'
probe.parent.mkdir(exist_ok=True)
probe.write_text(json.dumps({'classification':'file-transfer check, not model evidence','message':'download works'}, indent=2))
display(FileLink(str(probe)))
print('Download this file now. If the link is unsupported, use Kaggle’s working-files/output browser.')
print('Do not load the model until you have confirmed download access.')


## Load the pinned model after the download check
Outputs are written beneath the extracted project's results/ in /kaggle/working. This is session storage, NOT a mounted persistent Drive. Download every response before importing it on Mac, and download a complete evidence backup before ending the session. Do not assume a notebook autosave retains runtime files. Avoid Save & Run All, which would re-execute generations.


In [ ]:
from datetime import datetime, timezone
from qwen_adapter.settings import validate
from qwen_adapter.thinking_model import ThinkingQwen
assert torch.cuda.is_available(), 'GPU lost; stop before loading'
assert 'engine' not in globals(), 'Model already exists; do not load twice'
config = validate(json.loads((ROOT/CONFIG_NAME).read_text()))
save_dir = ROOT/'results'/('kaggle_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
save_dir.mkdir(parents=True, exist_ok=False)
(save_dir/'PACKAGE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
(save_dir/'package_identity.json').write_text(json.dumps(package_identity, indent=2))
(save_dir/'config.json').write_text(json.dumps(config, indent=2))
# Verify the spool filesystem supports atomic hard-link publication before model loading.
import os, tempfile
with tempfile.TemporaryDirectory(dir=ROOT/'work') as folder:
    first = Path(folder)/'first'; first.write_bytes(b'filesystem check only')
    second = Path(folder)/'second'; os.link(first, second)
    assert first.stat().st_ino == second.stat().st_ino
engine = ThinkingQwen(config['agent']['model'], config['agent']['qwen_options']['generation'])
assert engine.metadata['resolved_revision'] == config['agent']['qwen_options']['generation']['revision']
assert engine.metadata['native_thinking_flag'] is True
assert engine.metadata['template_sha256'] == 'a55ee1b1660128b7098723e0abcd92caa0788061051c62d51cbe87d9cf1974d8'
engine.metadata.update(runtime_platform='kaggle_manual_v1', cuda_runtime=torch.version.cuda,
    visible_gpu_names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], active_gpu_index=0)
(save_dir/'model_metadata.json').write_text(json.dumps(engine.metadata, indent=2))
print(json.dumps(engine.metadata, indent=2))
print('Session evidence:', save_dir)


## Define exchange helper — no generation in this cell
Requests are pasted as data literals produced by scripts/kaggle_request_cell.py on the Mac. Do not manually quote or edit JSON. Downloads use file links or Kaggle’s working-files browser, not google.colab.files. If this kernel restarts, inspect saved/downloaded evidence before any generation; do not regenerate a claimed call.


In [ ]:
from qwen_adapter.manual_exchange import generate_bundle
from qwen_adapter.protocol import parse_generation
from qwen_adapter.spool import digest
import uuid, copy

def evaluate_and_save(request):
    cid = request['call_id']
    bundle = generate_bundle(request, engine, ROOT/'work/manual_spool', retained_dir=save_dir/'records')
    bundle_path = save_dir/(cid + '.response_bundle.json')
    serialized = json.dumps(bundle, ensure_ascii=False, indent=2)
    if bundle_path.exists():
        assert bundle_path.read_text() == serialized, 'Conflicting existing bundle; stop'
    else:
        with bundle_path.open('x') as stream:
            stream.write(serialized)
    checksum = hashlib.sha256(serialized.encode()).hexdigest()
    checksum_path = bundle_path.with_suffix('.json.sha256')
    if not checksum_path.exists():
        checksum_path.write_text(checksum + '  ' + bundle_path.name + '\n')
    print('Call:', cid)
    print('Error:', bundle['response'].get('error'))
    print('Output:', bundle['response'].get('text'))
    raw = bundle.get('generation') or {}
    print('Tokens:', len(raw.get('input_token_ids', [])), len(raw.get('generated_token_ids', [])))
    print('Seconds:', raw.get('elapsed_seconds'), 'peak allocated bytes:', raw.get('peak_allocated_bytes'))
    print('Bundle SHA256:', checksum)
    display(FileLink(str(bundle_path)))
    print('Download this exact bundle before Mac import. File-browser path:', bundle_path)
    if bundle['response'].get('error'):
        raise RuntimeError('STOP: download failure evidence; no retry')
    parsed = parse_generation(raw, engine.settings)
    for position, capture in raw['activations'].items():
        print(position, 'replay:', capture['replay_passed'], 'dimensions:', {k:len(v) for k,v in capture['vectors'].items()})
    return bundle, parsed


## One platform compatibility smoke
No shell execution. Run once, download the result and inspect reasoning closure, final_report and P0/R1 replay. A pass is not task success. Do not repeat the old Docker diagnostic suite. Start a fresh Mac small-source controller only after this passes and prior interrupted run is stopped.


In [ ]:
if 'smoke_request' not in globals():
    smoke_request = {'call_id':uuid.uuid4().hex, 'model':engine.model_id, 'settings':copy.deepcopy(engine.settings),
        'messages':[{'role':'system','content':'You are testing a reasoning interface. Do not execute shell commands.'},
                    {'role':'user','content':'What is 7 + 8?'}],
        'tools':[{'type':'function','function':{'name':'execute_command','description':'Execute a shell command on the Mac.',
            'parameters':{'type':'object','properties':{'command':{'type':'string'}},'required':['command']}}}]}
    smoke_request['request_hash'] = digest(smoke_request)
smoke_bundle, smoke_parsed = evaluate_and_save(smoke_request)
assert smoke_parsed['kind'] == 'report', 'STOP: no shell action in smoke'
print('Compatibility smoke passed; review the report and download the bundle before starting the Mac task.')


## Actual task: paste each Mac-generated request cell below
On Mac export the pending request, then run scripts/kaggle_request_cell.py --run-dir RUN --request REQUEST.json | pbcopy. Paste the clipboard into a new Kaggle code cell and run once. Download the returned CALL.response_bundle.json and import it with scripts/manual_exchange.py. Importing releases Docker execution on Mac. Leave the smoke cell untouched. Repeat only until final_report or the first technical failure/cap; no coaching or repairs.


In [ ]:
# Paste the complete cell emitted by scripts/kaggle_request_cell.py here.
# Do not paste raw JSON or rerun the smoke cell for a task request.


## Evidence backup — no generation
Run after completed calls and before ending the session. Download this archive AND each response bundle. File links are only usable while the notebook backend is available. Kaggle saved-output/version retention must be verified in the UI; it is not a replacement for the Mac copies.


In [ ]:
backup_dir = ROOT/'work/backups'
backup_dir.mkdir(parents=True, exist_ok=True)
backup_path = backup_dir/(save_dir.name + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.zip')
with zipfile.ZipFile(backup_path, 'x', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(save_dir.rglob('*')):
        if path.is_file():
            archive.write(path, str(path.relative_to(save_dir.parent)))
print('Backup SHA256:', hashlib.sha256(backup_path.read_bytes()).hexdigest())
display(FileLink(str(backup_path)))
print('Download before disconnecting:', backup_path)
